In [ ]:
pip install pandas openpyxl xlwings

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 17.6 MB/s eta 0:00:00


In [ ]:
import numpy as np
import pandas as pd
import openpyxl

In [ ]:
#Classification of Results truth file
from google.colab import files
LLM_review = files.upload()

Saving Classification of the Results.xlsx to Classification of the Results.xlsx


In [ ]:
ground_truth = files.upload()

Saving Ground Truth Review.xlsx to Ground Truth Review.xlsx


In [ ]:
LLM_review_file = pd.read_excel('Classification of the Results.xlsx', sheet_name='LLM Review Score')
LLM_review_file

,LLM,Prompting Strategies,Name of the System,Review Criterion,Run,Score given by the LLM
0,GPT4o,Zero Shot,Baidu Apollo,Argument Comprehension,1,2.0
1,GPT4o,Zero Shot,Baidu Apollo,Argument Comprehension,2,2.0
2,GPT4o,Zero Shot,Baidu Apollo,Argument Comprehension,3,2.0
3,GPT4o,Zero Shot,Baidu Apollo,Argument Comprehension,4,2.0
4,GPT4o,Zero Shot,Baidu Apollo,Argument Comprehension,5,2.0
...,...,...,...,...,...,...
955,Gemini 2.0 Flash,CoT One Shot,Lane Management System,Argument Criticism and Defeat,1,3.0
956,Gemini 2.0 Flash,CoT One Shot,Lane Management System,Argument Criticism and Defeat,2,3.0
957,Gemini 2.0 Flash,CoT One Shot,Lane Management System,Argument Criticism and Defeat,3,3.0
958,Gemini 2.0 Flash,CoT One Shot,Lane Management System,Argument Criticism and Defeat,4,3.0


In [ ]:
def convert_to_likert(file):
  for index, row in file.iterrows():
    if (row['Score given by the LLM'] <= 5.0) & (row['Score given by the LLM'] > 3.0):
      file.loc[index, 'Likert Score'] = 'Not Acceptable'
    elif (row['Score given by the LLM'] <= 3.0) & (row['Score given by the LLM'] >= 1.0):
      file.loc[index,'Likert Score'] = 'Acceptable'
  return file


likert_LLM_file = convert_to_likert(LLM_review_file)
likert_LLM_file

,LLM,Prompting Strategies,Name of the System,Review Criterion,Run,Score given by the LLM,Likert Score
0,GPT4o,Zero Shot,Baidu Apollo,Argument Comprehension,1,2.0,Acceptable
1,GPT4o,Zero Shot,Baidu Apollo,Argument Comprehension,2,2.0,Acceptable
2,GPT4o,Zero Shot,Baidu Apollo,Argument Comprehension,3,2.0,Acceptable
3,GPT4o,Zero Shot,Baidu Apollo,Argument Comprehension,4,2.0,Acceptable
4,GPT4o,Zero Shot,Baidu Apollo,Argument Comprehension,5,2.0,Acceptable
...,...,...,...,...,...,...,...
955,Gemini 2.0 Flash,CoT One Shot,Lane Management System,Argument Criticism and Defeat,1,3.0,Acceptable
956,Gemini 2.0 Flash,CoT One Shot,Lane Management System,Argument Criticism and Defeat,2,3.0,Acceptable
957,Gemini 2.0 Flash,CoT One Shot,Lane Management System,Argument Criticism and Defeat,3,3.0,Acceptable
958,Gemini 2.0 Flash,CoT One Shot,Lane Management System,Argument Criticism and Defeat,4,3.0,Acceptable


In [ ]:
#Ground truth Review file
ground_truth_file = pd.read_excel('Ground Truth Review.xlsx', 'Summary')
ground_truth_file


,Assurance Case,Review Criterion,Score
0,GPCA,Argument Comprehension,4
1,GPCA,Well-Formedness (Syntax),5
2,GPCA,Expressive Sufficiency,2
3,GPCA,Argument Criticism and Defeat,3
4,IM Software,Argument Comprehension,4
5,IM Software,Well-Formedness (Syntax),5
6,IM Software,Expressive Sufficiency,2
7,IM Software,Argument Criticism and Defeat,4
8,Baidu Apollo,Argument Comprehension,2
9,Baidu Apollo,Well-Formedness (Syntax),2


In [ ]:
def convert_ground(file):
  for index, row in file.iterrows():
    if (row['Score'] <= 5) & (row['Score'] > 3):
      file.loc[index, 'Likert Score'] = 'Not Acceptable'
    elif (row['Score'] <= 3) & (row['Score'] >= 1):
      file.loc[index,'Likert Score'] = 'Acceptable'

  return file

likert_ground_truth_file = convert_ground(ground_truth_file)
likert_ground_truth_file

,Assurance Case,Review Criterion,Score,Likert Score
0,GPCA,Argument Comprehension,4,Not Acceptable
1,GPCA,Well-Formedness (Syntax),5,Not Acceptable
2,GPCA,Expressive Sufficiency,2,Acceptable
3,GPCA,Argument Criticism and Defeat,3,Acceptable
4,IM Software,Argument Comprehension,4,Not Acceptable
5,IM Software,Well-Formedness (Syntax),5,Not Acceptable
6,IM Software,Expressive Sufficiency,2,Acceptable
7,IM Software,Argument Criticism and Defeat,4,Not Acceptable
8,Baidu Apollo,Argument Comprehension,2,Acceptable
9,Baidu Apollo,Well-Formedness (Syntax),2,Acceptable


GPCA

In [ ]:
GPCA_likert_LLM = likert_LLM_file.loc[likert_LLM_file['Name of the System'] == 'GPCA']
GPCA_likert_ground = likert_ground_truth_file.loc[likert_ground_truth_file['Assurance Case'] == 'GPCA']
#print(GPCA_likert_LLM)
#GPCA_likert_ground

GPCA_TP = 0
GPCA_FN = 0
GPCA_FP = 0

#Argument Comprehension
GPCA_argument = GPCA_likert_LLM.loc[GPCA_likert_LLM['Review Criterion'] == 'Argument Comprehension']
GPCA_ground_argument = GPCA_likert_ground.loc[GPCA_likert_ground['Review Criterion'] == 'Argument Comprehension']
for index, row in GPCA_argument.iterrows():
  if (row['Likert Score'] == 'Acceptable') and (GPCA_ground_argument['Likert Score'].to_string(index=False) == 'Acceptable'):
    GPCA_TP += 1
  elif (row['Likert Score'] == 'Not Acceptable') and (GPCA_ground_argument['Likert Score'].to_string(index=False) == 'Acceptable'):
    GPCA_FN += 1
  elif (row['Likert Score'] == 'Acceptable') and (GPCA_ground_argument['Likert Score'].to_string(index=False) == 'Not Acceptable'):
    GPCA_FP += 1


#print(GPCA_ground_argument)
#print(GPCA_TP)
#print(GPCA_FN)

#Well-Formedness (Syntax)
GPCA_well_formedness = GPCA_likert_LLM.loc[GPCA_likert_LLM['Review Criterion'] == 'Well-Formedness(Syntax)']
GPCA_ground_well_formedness = GPCA_likert_ground.loc[GPCA_likert_ground['Review Criterion'] == 'Well-Formedness(Syntax)']
for index, row in GPCA_well_formedness.iterrows():
  if (row['Likert Score'] == 'Acceptable') and (GPCA_ground_well_formedness['Likert Score'].to_string(index=False) == 'Acceptable'):
    GPCA_TP += 1
  elif (row['Likert Score'] == 'Not Acceptable') and (GPCA_ground_well_formedness['Likert Score'].to_string(index=False) == 'Acceptable'):
    GPCA_FN += 1
  elif (row['Likert Score'] == 'Acceptable') and (GPCA_ground_well_formedness['Likert Score'].to_string(index=False) == 'Not Acceptable'):
    GPCA_FP += 1


#print(GPCA_ground_well_formedness)
#print(GPCA_TP)
#print(GPCA_FN)

#Expressive Sufficiency
GPCA_expressive = GPCA_likert_LLM.loc[GPCA_likert_LLM['Review Criterion'] == 'Expressive Sufficiency ']
GPCA_ground_expressive = GPCA_likert_ground.loc[GPCA_likert_ground['Review Criterion'] == 'Expressive Sufficiency']
for index, row in GPCA_expressive.iterrows():
  if (row['Likert Score'] == 'Acceptable') and (GPCA_ground_expressive['Likert Score'].to_string(index=False) == 'Acceptable'):
    GPCA_TP += 1
  elif (row['Likert Score'] == 'Not Acceptable') and (GPCA_ground_expressive['Likert Score'].to_string(index=False) == 'Acceptable'):
    GPCA_FN += 1
  elif (row['Likert Score'] == 'Acceptable') and (GPCA_ground_expressive['Likert Score'].to_string(index=False) == 'Not Acceptable'):
    GPCA_FP += 1

#print(GPCA_ground_expressive)
#print(GPCA_TP)
#print(GPCA_FN)

#Argument Criticism and Defeat
GPCA_criticism = GPCA_likert_LLM.loc[GPCA_likert_LLM['Review Criterion'] == 'Argument Criticism and Defeat']
GPCA_ground_criticism = GPCA_likert_ground.loc[GPCA_likert_ground['Review Criterion'] == 'Argument Criticism and Defeat']
for index, row in GPCA_criticism.iterrows():
  if (row['Likert Score'] == 'Acceptable') and (GPCA_ground_criticism['Likert Score'].to_string(index=False) == 'Acceptable'):
    GPCA_TP += 1
  elif (row['Likert Score'] == 'Not Acceptable') and (GPCA_ground_criticism['Likert Score'].to_string(index=False) == 'Acceptable'):
    GPCA_FN += 1
  elif (row['Likert Score'] == 'Acceptable') and (GPCA_ground_criticism['Likert Score'].to_string(index=False) == 'Not Acceptable'):
    GPCA_FP += 1

#print(GPCA_ground_criticism)
#print(GPCA_TP)
#print(GPCA_FN)


#Recall
GPCA_recall = GPCA_TP / (GPCA_TP + GPCA_FN)
print("Recall:",GPCA_recall)


#Precision
GPCA_precision = GPCA_TP / (GPCA_TP + GPCA_FP)
print("Precision:",GPCA_precision)


#F1 Score
GPCA_F1 = 2 * (GPCA_precision * GPCA_recall) / (GPCA_precision + GPCA_recall)
print("F1 Score:",GPCA_F1)

Recall: 0.5916666666666667
Precision: 0.6454545454545455
F1 Score: 0.6173913043478261


IM Server Software

In [ ]:
IM_likert_LLM = likert_LLM_file.loc[likert_LLM_file['Name of the System'] == 'IM Software']
IM_likert_ground = likert_ground_truth_file.loc[likert_ground_truth_file['Assurance Case'] == 'IM Software']
#print(IM_likert_ground)

IM_TP = 0
IM_FN = 0
IM_FP = 0

#Argument Comprehension
IM_argument = IM_likert_LLM.loc[IM_likert_LLM['Review Criterion'] == 'Argument Comprehension']
IM_argument_ground = IM_likert_ground.loc[IM_likert_ground['Review Criterion'] == 'Argument Comprehension']
for index, rows in IM_argument.iterrows():
  if (row['Likert Score'] == 'Acceptable') and (IM_argument_ground['Likert Score'].to_string(index=False) == 'Acceptable'):
    IM_TP += 1
  elif (row['Likert Score'] == 'Not Acceptable') and (IM_argument_ground['Likeert Score'].to_string(index=False) == 'Acceptable'):
    IM_FN += 1
  elif (row['Likert Score'] == 'Acceptable') and (IM_argument_ground['Likert Score'].to_string(index=False) == 'Not Acceptable'):
    IM_FP += 1

#print(IM_argument_ground)
#print(IM_TP)
#print(IM_FN)


#Well-Formedness
IM_well_formedness = IM_likert_LLM.loc[IM_likert_LLM['Review Criterion'] == 'Well-Formedness(Syntax)']
IM_well_formedness_ground = IM_likert_ground.loc[IM_likert_ground['Review Criterion'] == 'Well-Formedness (Syntax)']
for index, rows in IM_argument.iterrows():
  if (row['Likert Score'] == 'Acceptable') and (IM_well_formedness_ground['Likert Score'].to_string(index=False) == 'Acceptable'):
    IM_TP += 1
  elif (row['Likert Score'] == 'Not Acceptable') and (IM_well_formedness_ground['Likeert Score'].to_string(index=False) == 'Acceptable'):
    IM_FN += 1
  elif (row['Likert Score'] == 'Acceptable') and (IM_well_formedness_ground['Likert Score'].to_string(index=False) == 'Not Acceptable'):
    IM_FP += 1

#print(IM_well_formedness_ground)
#print(IM_TP)
#print(IM_FN)

#Expressive Sufficiency
IM_expressive = IM_likert_LLM.loc[IM_likert_LLM['Review Criterion'] == 'Expressive Sufficiency ']
IM_expressive_ground = IM_likert_ground.loc[IM_likert_ground['Review Criterion'] == 'Expressive Sufficiency']
for index, rows in IM_expressive.iterrows():
  if (row['Likert Score'] == 'Acceptable') and (IM_expressive_ground['Likert Score'].to_string(index=False) == 'Acceptable'):
    IM_TP += 1
  elif (row['Likert Score'] == 'Not Acceptable') and (IM_expressive_ground['Likeert Score'].to_string(index=False) == 'Acceptable'):
    IM_FN += 1
  elif (row['Likert Score'] == 'Acceptable') and (IM_expressive_ground['Likert Score'].to_string(index=False) == 'Not Acceptable'):
    IM_FP += 1

#print(IM_expressive_ground)
#print(IM_TP)
#print(IM_FN)


#Argument Criticism and Defeat
IM_criticism = IM_likert_LLM.loc[IM_likert_LLM['Review Criterion'] == 'Argument Criticism and Defeat']
IM_criticism_ground = IM_likert_ground.loc[IM_likert_ground['Review Criterion'] == 'Argument Criticism and Defeat']
for index, rows in IM_criticism.iterrows():
  if (row['Likert Score'] == 'Acceptable') and (IM_criticism_ground['Likert Score'].to_string(index=False) == 'Acceptable'):
    IM_TP += 1
  elif (row['Likert Score'] == 'Not Acceptable') and (IM_criticism_ground['Likeert Score'].to_string(index=False) == 'Acceptable'):
    IM_FN += 1
  elif (row['Likert Score'] == 'Acceptable') and (IM_expressive_ground['Likert Score'].to_string(index=False) == 'Not Acceptable'):
    IM_FP += 1

#print(IM_criticism_ground)
#print(IM_TP)
#print(IM_FN)


#Recall
IM_recall = IM_TP / (IM_TP + IM_FN)
print("Recall:",IM_recall)


#Precision
IM_precision = IM_TP / (IM_TP + IM_FP)
print("Precision:", IM_precision)


#F1 Score
IM_F1 = 2 * (IM_precision * IM_recall) / (IM_precision + IM_recall)
print("F1 Score:", IM_F1)

Recall: 1.0
Precision: 0.3333333333333333
F1 Score: 0.5


Baidu Apollo

In [ ]:
Baidu_likert = likert_LLM_file.loc[likert_LLM_file['Name of the System'] == 'Baidu Apollo']
Baidu_likert_ground = likert_ground_truth_file.loc[likert_ground_truth_file['Assurance Case'] == 'Baidu Apollo']


#print(Baidu_likert_ground)

Baidu_TP = 0
Baidu_FN = 0
Baidu_FP = 0

#Argument Comprehension
Baidu_argument = Baidu_likert.loc[Baidu_likert['Review Criterion'] == 'Argument Comprehension']
Baidu_argument_ground = Baidu_likert_ground.loc[Baidu_likert_ground['Review Criterion'] == 'Argument Comprehension']
for index, rows in Baidu_argument.iterrows():
  if (row['Likert Score'] == 'Acceptable') and (Baidu_argument_ground['Likert Score'].to_string(index=False) == 'Acceptable'):
    Baidu_TP += 1
  elif (row['Likert Score'] == 'Not Acceptable') and (Baidu_argument_ground['Likert Score'].to_string(index=False) == 'Acceptable'):
    Baidu_FN += 1
  elif (row['Likert Score'] == 'Acceptable') and (Baidu_argument_ground['Likert Score'].to_string(index=False) == 'Not Acceptable'):
    Baidu_FP += 1

#print(Baidu_argument_ground)
#print(Baidu_TP)
#print(Baidu_FN)


#Well-Formedness
Baidu_well_formedness = Baidu_likert.loc[Baidu_likert['Review Criterion'] == 'Well-Formedness(Syntax)']
Baidu_well_formedness_ground = Baidu_likert_ground.loc[Baidu_likert_ground['Review Criterion'] == 'Well-Formedness (Syntax)']
for index, rows in Baidu_well_formedness.iterrows():
  if (row['Likert Score'] == 'Acceptable') and (Baidu_well_formedness_ground['Likert Score'].to_string(index=False) == 'Acceptable'):
    Baidu_TP += 1
  elif (row['Likert Score'] == 'Not Acceptable') and (Baidu_well_formedness_ground['Likert Score'].to_string(index=False) == 'Acceptable'):
    Baidu_FN += 1
  elif (row['Likert Score'] == 'Acceptable') and (Baidu_well_formedness_ground['Likert Score'].to_string(index=False) == 'Not Acceptable'):
    Baidu_FP += 1

#print(Baidu_well_formedness_ground)
#print(Baidu_TP)
#print(Baidu_FN)

#Expressive Sufficiency
Baidu_expressive = Baidu_likert.loc[Baidu_likert['Review Criterion'] == 'Expressive Sufficiency ']
Baidu_expressive_ground = Baidu_likert_ground.loc[Baidu_likert_ground['Review Criterion'] == 'Expressive Sufficiency']
for index, rows in Baidu_expressive.iterrows():
  if (row['Likert Score'] == 'Acceptable') and (Baidu_expressive_ground['Likert Score'].to_string(index=False) == 'Acceptable'):
    Baidu_TP += 1
  elif (row['Likert Score'] == 'Not Acceptable') and (Baidu_expressive_ground['Likert Score'].to_string(index=False) == 'Acceptable'):
    Baidu_FN += 1
  elif (row['Likert Score'] == 'Acceptable') and (Baidu_expressive_ground['Likert Score'].to_string(index=False) == 'Not Acceptable'):
    Baidu_FP += 1

#print(Baidu_expressive_ground)
#print(Baidu_TP)
#print(Baidu_FN)



#Argument Criticism and Defeat
Baidu_criticism = Baidu_likert.loc[Baidu_likert['Review Criterion'] == 'Argument Criticism and Defeat']
Baidu_criticism_ground = Baidu_likert_ground.loc[Baidu_likert_ground['Review Criterion'] == 'Argument Criticism and Defeat']
for index, rows in Baidu_criticism.iterrows():
  if (row['Likert Score'] == 'Acceptable') and (Baidu_criticism_ground['Likert Score'].to_string(index=False) == 'Acceptable'):
    Baidu_TP += 1
  elif (row['Likert Score'] == 'Not Acceptable') and (Baidu_criticism_ground['Likert Score'].to_string(index=False) == 'Acceptable'):
    Baidu_FN += 1
  elif (row['Likert Score'] == 'Acceptable') and (Baidu_criticism_ground['Likert Score'].to_string(index=False) == 'Not Acceptable'):
    Baidu_FP += 1

#print(Baidu_criticism_ground)
#print(Baidu_TP)
#print(Baidu_FN)


#Recall
Baidu_recall = Baidu_TP / (Baidu_TP + Baidu_FN)
print("Recall:",Baidu_recall)



#Precision
Baidu_precision = Baidu_TP / (Baidu_TP + Baidu_FP)
print("Precision:", Baidu_precision)



#F1 Score
Baidu_F1 = 2 * (Baidu_precision * Baidu_recall) / (Baidu_precision + Baidu_recall)
print("F1 Score:", Baidu_F1)

Recall: 1.0
Precision: 1.0
F1 Score: 1.0


Lane Management System

In [ ]:
LMS_likert = likert_LLM_file.loc[likert_LLM_file['Name of the System'] == 'Lane Management System']
LMS_likert_ground = likert_ground_truth_file.loc[likert_ground_truth_file['Assurance Case'] == 'Lane Management System']

#print(LMS_likert_ground)


LMS_TP = 0
LMS_FN = 0
LMS_FP = 0


#Argument Comprehension
LMS_argument = LMS_likert.loc[LMS_likert['Review Criterion'] == 'Argument Comprehension']
LMS_argument_ground = LMS_likert_ground.loc[LMS_likert_ground['Review Criterion'] == 'Argument Comprehension']
for index, rows in LMS_argument.iterrows():
  if (row['Likert Score'] == 'Acceptable') and (LMS_argument_ground['Likert Score'].to_string(index=False) == 'Acceptable'):
    LMS_TP += 1
  elif (row['Likert Score'] == 'Not Acceptable') and (LMS_argument_ground['Likert Score'].to_string(index=False) == 'Acceptable'):
    LMS_FN += 1
  elif (row['Likert Score'] == 'Acceptable') and (LMS_argument_ground['Likert Score'].to_string(index=False) == 'Not Acceptable'):
    LMS_FP += 1

#print(LMS_argument_ground)
#print(LMS_TP)
#print(LMS_FN)


#Well-Formedness
LMS_well_formedness = LMS_likert.loc[LMS_likert['Review Criterion'] == 'Well-Formedness(Syntax)']
LMS_well_formedness_ground = LMS_likert_ground.loc[LMS_likert_ground['Review Criterion'] == 'Well-Formedness (Syntax)']
for index, rows in LMS_well_formedness.iterrows():
  if (row['Likert Score'] == 'Acceptable') and (LMS_well_formedness_ground['Likert Score'].to_string(index=False) == 'Acceptable'):
    LMS_TP += 1
  elif (row['Likert Score'] == 'Not Acceptable') and (LMS_well_formedness_ground['Likert Score'].to_string(index=False) == 'Acceptable'):
    LMS_FN += 1
  elif (row['Likert Score'] == 'Acceptable') and (LMS_well_formedness_ground['Likert Score'].to_string(index=False) == 'Not Acceptable'):
    LMS_FP += 1

#print(LMS_well_formedness_ground)
#print(LMS_TP)
#print(LMS_FN)


#Expressive Sufficiency
LMS_expressive = LMS_likert.loc[LMS_likert['Review Criterion'] == 'Expressive Sufficiency ']
LMS_expressive_ground = LMS_likert_ground.loc[LMS_likert_ground['Review Criterion'] == 'Expressive Sufficiency']
for index, rows in LMS_expressive.iterrows():
  if (row['Likert Score'] == 'Acceptable') and (LMS_expressive_ground['Likert Score'].to_string(index=False) == 'Acceptable'):
    LMS_TP += 1
  elif (row['Likert Score'] == 'Not Acceptable') and (LMS_expressive_ground['Likert Score'].to_string(index=False) == 'Acceptable'):
    LMS_FN += 1
  elif (row['Likert Score'] == 'Acceptable') and (LMS_expressive_ground['Likert Score'].to_string(index=False) == 'Not Acceptable'):
    LMS_FP += 1

#print(LMS_expressive_ground)
#print(LMS_TP)
#print(LMS_FN)



#Argument Criticism and Defeat
LMS_criticism = LMS_likert.loc[LMS_likert['Review Criterion'] == 'Argument Criticism and Defeat']
LMS_criticism_ground = LMS_likert_ground.loc[LMS_likert_ground['Review Criterion'] == 'Argument Criticism and Defeat']
for index, rows in LMS_criticism.iterrows():
  if (row['Likert Score'] == 'Acceptable') and (LMS_criticism_ground['Likert Score'].to_string(index=False) == 'Acceptable'):
    LMS_TP += 1
  elif (row['Likert Score'] == 'Not Acceptable') and (LMS_criticism_ground['Likert Score'].to_string(index=False) == 'Acceptable'):
    LMS_FN += 1
  elif (row['Likert Score'] == 'Acceptable') and (LMS_criticism_ground['Likert Score'].to_string(index=False) == 'Not Acceptable'):
    LMS_FP += 1


#print(LMS_criticism_ground)
#print(LMS_TP)
#print(LMS_FN)


#Recall
LMS_recall = LMS_TP / (LMS_TP + LMS_FN)
print("Recall:",LMS_recall)




#Precision
LMS_precision = LMS_TP / (LMS_TP + LMS_FP)
print("Precision:", LMS_precision)



#F1 Score
LMS_F1 = 2 * (LMS_precision * LMS_recall) / (LMS_precision + LMS_recall)
print("F1 Score:", LMS_F1)

Recall: 1.0
Precision: 1.0
F1 Score: 1.0


LLM

GPT4o

In [ ]:
def calculate_LLM(assurance_case, review_criterion, review_criterion_ground, LLM):
  TP = 0
  FN = 0
  FP = 0
  temp = likert_LLM_file.loc[likert_LLM_file['Name of the System'] == assurance_case]
  temp_ground = likert_ground_truth_file.loc[likert_ground_truth_file['Assurance Case'] == assurance_case]

  temp_LLM = temp.loc[temp['LLM'] == LLM]


  temp_criterion = temp_LLM.loc[temp_LLM['Review Criterion'] == review_criterion]
  temp_ground_criterion = temp_ground.loc[temp_ground['Review Criterion'] == review_criterion_ground]

  for index, row in temp_criterion.iterrows():
    if (row['Likert Score'] == 'Acceptable') and (temp_ground_criterion['Likert Score'].to_string(index=False) == 'Acceptable'):
      TP += 1
    elif (row['Likert Score'] == 'Not Acceptable') and (temp_ground_criterion['Likert Score'].to_string(index=False) == 'Acceptable'):
      FN += 1
    elif (row['Likert Score'] == 'Acceptable') and (temp_ground_criterion['Likert Score'].to_string(index=False) == 'Not Acceptable'):
      FP += 1


  return TP, FN, FP


###########GPT4o/GPCA

GPT4o_GPCA_TP = 0
GPT4o_GPCA_FN = 0
GPT4o_GPCA_FP = 0

#Argument Comprehension
TP , FN , FP = calculate_LLM('GPCA', 'Argument Comprehension', 'Argument Comprehension', 'GPT4o')
GPT4o_GPCA_TP += TP
GPT4o_GPCA_FN += FN
GPT4o_GPCA_FP += FP

#print(GPT4o_GPCA_TP)
#print(GPT4o_GPCA_FN)
#print(GPT4o_GPCA_FP)


#Well-Formedness
TP , FN , FP= calculate_LLM('GPCA', 'Well-Formedness(Syntax)', 'Well-Formedness (Syntax)', 'GPT4o')
GPT4o_GPCA_TP += TP
GPT4o_GPCA_FN += FN
GPT4o_GPCA_FP += FP

#print(GPT4o_GPCA_TP)
#print(GPT4o_GPCA_FN)
#print(GPT4o_GPCA_FP)

#Expressive Sufficiency
TP , FN , FP = calculate_LLM('GPCA', 'Expressive Sufficiency ', 'Expressive Sufficiency', 'GPT4o')
GPT4o_GPCA_TP += TP
GPT4o_GPCA_FN += FN
GPT4o_GPCA_FP += FP

#print(GPT4o_GPCA_TP)
#print(GPT4o_GPCA_FN)
#print(GPT4o_GPCA_FP)


#Argument Criticism and Defeat
TP , FN , FP = calculate_LLM('GPCA', 'Argument Criticism and Defeat', 'Argument Criticism and Defeat', 'GPT4o')
GPT4o_GPCA_TP += TP
GPT4o_GPCA_FN += FN
GPT4o_GPCA_FP += FP

#print(GPT4o_GPCA_TP)
#print(GPT4o_GPCA_FN)
#print(GPT4o_GPCA_FP)

#Recall
GPT4o_GPCA_recall = GPT4o_GPCA_TP / (GPT4o_GPCA_TP + GPT4o_GPCA_FN)
print("GPCA GPT4o Recall:",GPT4o_GPCA_recall)
print("Recall Percent:",round(GPT4o_GPCA_recall * 100, 2))


#Precision
GPT4o_GPCA_precision = GPT4o_GPCA_TP / (GPT4o_GPCA_TP + GPT4o_GPCA_FP)
print("GPCA GPT4o Precision:", GPT4o_GPCA_precision)
print("Precision Percent:", round(GPT4o_GPCA_precision * 100, 2))


#F1 Score
GPT4o_GPCA_F1 = 2 * (GPT4o_GPCA_precision * GPT4o_GPCA_recall) / (GPT4o_GPCA_precision + GPT4o_GPCA_recall)
print("GPCA GPT4o F1 Score:", GPT4o_GPCA_F1)
print("F1 Score Percent:", round(GPT4o_GPCA_F1 * 100, 2))


############GPT4o/IM Server Software
GPT4o_IM_TP = 0
GPT4o_IM_FN = 0
GPT4o_IM_FP = 0

#Argument Comprehension
TP , FN , FP = calculate_LLM('IM Software', 'Argument Comprehension', 'Argument Comprehension', 'GPT4o')
GPT4o_IM_TP += TP
GPT4o_IM_FN += FN
GPT4o_IM_FP += FP

#Well-Formedness
TP , FN , FP = calculate_LLM('IM Software', 'Well-Formedness(Syntax)', 'Well-Formedness (Syntax)', 'GPT4o')
GPT4o_IM_TP += TP
GPT4o_IM_FN += FN
GPT4o_IM_FP += FP

#Expressive Sufficiency
TP , FN , FP = calculate_LLM('IM Software', 'Expressive Sufficiency ', 'Expressive Sufficiency', 'GPT4o')
GPT4o_IM_TP += TP
GPT4o_IM_FN += FN
GPT4o_IM_FP += FP

#Argument Criticism and Defeat
TP , FN , FP = calculate_LLM('IM Software', 'Argument Criticism and Defeat', 'Argument Criticism and Defeat', 'GPT4o')
GPT4o_IM_TP += TP
GPT4o_IM_FN += FN
GPT4o_IM_FP += FP


#Recall
GPT4o_IM_recall = GPT4o_IM_TP / (GPT4o_IM_TP + GPT4o_IM_FN)
print("IM Software GPT4o Recall:",GPT4o_IM_recall)
print("IM Software GPT4o Recall Percent:",round(GPT4o_IM_recall * 100, 2))

#Precision
GPT4o_IM_precision = GPT4o_IM_TP / (GPT4o_IM_TP + GPT4o_IM_FP)
print("IM Software GPT4o Precision:", GPT4o_IM_precision)
print("IM Software GPT4o Precision Percent:", round(GPT4o_IM_precision * 100, 2))


#F1 Score
GPT4o_IM_F1 = 2 * (GPT4o_IM_precision * GPT4o_IM_recall) / (GPT4o_IM_precision + GPT4o_IM_recall)
print("IM Software GPT4o F1:", GPT4o_IM_F1)
print("IM Software GPT4o F1 Percent:", round(GPT4o_IM_F1 * 100, 2))

############GPT4o/Baidu Apollo
GPT4o_Baidu_TP = 0
GPT4o_Baidu_FN = 0
GPT4o_Baidu_FP = 0

#Argument Comprehension
TP , FN , FP= calculate_LLM('Baidu Apollo', 'Argument Comprehension', 'Argument Comprehension', 'GPT4o')
GPT4o_Baidu_TP += TP
GPT4o_Baidu_FN += FN
GPT4o_Baidu_FP += FP

#Well-Formedness
TP , FN , FP= calculate_LLM('Baidu Apollo', 'Well-Formedness(Syntax)', 'Well-Formedness (Syntax)', 'GPT4o')
GPT4o_Baidu_TP += TP
GPT4o_Baidu_FN += FN
GPT4o_Baidu_FP += FP

#Expressive Sufficiency
TP , FN , FP= calculate_LLM('Baidu Apollo', 'Expressive Sufficiency ', 'Expressive Sufficiency', 'GPT4o')
GPT4o_Baidu_TP += TP
GPT4o_Baidu_FN += FN
GPT4o_Baidu_FP += FP

#Argument Criticism and Defeat
TP , FN , FP= calculate_LLM('Baidu Apollo', 'Argument Criticism and Defeat', 'Argument Criticism and Defeat', 'GPT4o')
GPT4o_Baidu_TP += TP
GPT4o_Baidu_FN += FN
GPT4o_Baidu_FP += FP

#Recall
GPT4o_Baidu_recall = GPT4o_Baidu_TP / (GPT4o_Baidu_TP + GPT4o_Baidu_FN)
print("Baidu Apollo GPT4o Recall:",GPT4o_Baidu_recall)
print("Baidu Apollo GPT4o Recall Percent:",round(GPT4o_Baidu_recall * 100, 2))

#Precision
GPT4o_Baidu_precision = GPT4o_Baidu_TP / (GPT4o_Baidu_TP + GPT4o_Baidu_FP)
print("Baidu Apollo GPT4o Prcision:", GPT4o_Baidu_precision)
print("Baidu Apollo GPT4o Prcision Percent:", round(GPT4o_Baidu_precision * 100, 2))


#F1 Score
GPT4o_Baidu_F1 = 2 * (GPT4o_Baidu_precision * GPT4o_Baidu_recall) / (GPT4o_Baidu_precision + GPT4o_Baidu_recall)
print("Baidu Apollo GPT4o F1 Score:", GPT4o_Baidu_F1)
print("Baidu Apollo GPT4o F1 Score Percent:", round(GPT4o_Baidu_F1 * 100, 2))

############GPT4o/Lane Management System
GPT4o_LMS_TP = 0
GPT4o_LMS_FN = 0
GPT4o_LMS_FP = 0

#Argument Comprehension
TP , FN , FP = calculate_LLM('Lane Management System', 'Argument Comprehension', 'Argument Comprehension', 'GPT4o')
GPT4o_LMS_TP += TP
GPT4o_LMS_FN += FN
GPT4o_LMS_FP += FP


#Well-Formedness
TP , FN , FP = calculate_LLM('Lane Management System', 'Well-Formedness(Syntax)', 'Well-Formedness (Syntax)', 'GPT4o')
GPT4o_LMS_TP += TP
GPT4o_LMS_FN += FN
GPT4o_LMS_FP += FP


#Expressive Sufficiency
TP , FN , FP = calculate_LLM('Lane Management System', 'Expressive Sufficiency ', 'Expressive Sufficiency', 'GPT4o')
GPT4o_LMS_TP += TP
GPT4o_LMS_FN += FN
GPT4o_LMS_FP += FP

#Argument Criticism and Defeat
TP , FN , FP= calculate_LLM('Lane Management System', 'Argument Criticism and Defeat', 'Argument Criticism and Defeat', 'GPT4o')
GPT4o_LMS_TP += TP
GPT4o_LMS_FN += FN
GPT4o_LMS_FP += FP

#Recall
GPT4o_LMS_recall = GPT4o_LMS_TP / (GPT4o_LMS_TP + GPT4o_LMS_FN)
print("LMS GPT4o Recall:",GPT4o_LMS_recall)
print("LMS GPT4o Recall Percent:",round(GPT4o_LMS_recall * 100, 2))

#Precision
GPT4o_LMS_precision = GPT4o_LMS_TP / (GPT4o_LMS_TP + GPT4o_LMS_FP)
print("LMS GPT4o Precision:", GPT4o_LMS_precision)
print("LMS GPT4o Precision Percent:", round(GPT4o_LMS_precision * 100, 2))


#F1 Score
GPT4o_LMS_F1 = 2 * (GPT4o_LMS_precision * GPT4o_LMS_recall) / (GPT4o_LMS_precision + GPT4o_LMS_recall)
print("LMS GPT4o F1 Score:", GPT4o_LMS_F1)
print("LMS GPT4o F1 Score Percent:", round(GPT4o_LMS_F1 * 100, 2))

GPCA GPT4o Recall: 0.7666666666666667
Recall Percent: 76.67
GPCA GPT4o Precision: 0.48936170212765956
Precision Percent: 48.94
GPCA GPT4o F1 Score: 0.5974025974025974
F1 Score Percent: 59.74
IM Software GPT4o Recall: 0.6
IM Software GPT4o Recall Percent: 60.0
IM Software GPT4o Precision: 0.20930232558139536
IM Software GPT4o Precision Percent: 20.93
IM Software GPT4o F1: 0.3103448275862069
IM Software GPT4o F1 Percent: 31.03
Baidu Apollo GPT4o Recall: 1.0
Baidu Apollo GPT4o Recall Percent: 100.0
Baidu Apollo GPT4o Prcision: 1.0
Baidu Apollo GPT4o Prcision Percent: 100.0
Baidu Apollo GPT4o F1 Score: 1.0
Baidu Apollo GPT4o F1 Score Percent: 100.0
LMS GPT4o Recall: 0.8833333333333333
LMS GPT4o Recall Percent: 88.33
LMS GPT4o Precision: 1.0
LMS GPT4o Precision Percent: 100.0
LMS GPT4o F1 Score: 0.9380530973451328
LMS GPT4o F1 Score Percent: 93.81


GPT4.1

In [ ]:

###############GPT4.1 GPCA

GPT4point1_GPCA_TP = 0
GPT4point1_GPCA_FN = 0
GPT4point1_GPCA_FP = 0

#Argument Comprehension
TP , FN , FP = calculate_LLM('GPCA', 'Argument Comprehension', 'Argument Comprehension', 'GPT4.1')
GPT4point1_GPCA_TP += TP
GPT4point1_GPCA_FN += FN
GPT4point1_GPCA_FP += FP

#Well-Formedness
TP , FN , FP = calculate_LLM('GPCA', 'Well-Formedness(Syntax)', 'Well-Formedness (Syntax)', 'GPT4.1')
GPT4point1_GPCA_TP += TP
GPT4point1_GPCA_FN += FN
GPT4point1_GPCA_FP += FP


#Expressive Sufficiency
TP , FN , FP = calculate_LLM('GPCA', 'Expressive Sufficiency ', 'Expressive Sufficiency', 'GPT4.1')
GPT4point1_GPCA_TP += TP
GPT4point1_GPCA_FN += FN
GPT4point1_GPCA_FP += FP

#Argument Criticism and Defeat
TP , FN , FP = calculate_LLM('GPCA', 'Argument Criticism and Defeat', 'Argument Criticism and Defeat', 'GPT4.1')
GPT4point1_GPCA_TP += TP
GPT4point1_GPCA_FN += FN
GPT4point1_GPCA_FP += FP

#Recall
GPT4point1_GPCA_recall = GPT4point1_GPCA_TP / (GPT4point1_GPCA_TP + GPT4point1_GPCA_FN)
print("GPCA Recall:", GPT4point1_GPCA_recall)
print("GPCA Recall Percent:", round(GPT4point1_GPCA_recall * 100, 2))

#Precision
GPT4point1_GPCA_precision = GPT4point1_GPCA_TP / (GPT4point1_GPCA_TP + GPT4point1_GPCA_FP)
print("GPCA Precision:", GPT4point1_GPCA_precision)
print("GPCA Precision Percent:", round(GPT4point1_GPCA_precision * 100, 2))


#F1 Score
GPT4point1_GPCA_F1 = 2 * (GPT4point1_GPCA_precision * GPT4point1_GPCA_recall) / (GPT4point1_GPCA_precision + GPT4point1_GPCA_recall)
print("GPCA F1:", GPT4point1_GPCA_F1)
print("GPCA F1 Percent:", round(GPT4point1_GPCA_F1 * 100, 2))


###############GPT4.1 IM Software
GPT4point1_IM_TP = 0
GPT4point1_IM_FN = 0
GPT4point1_IM_FP = 0

#Argument Comprehension
TP , FN , FP = calculate_LLM('IM Software', 'Argument Comprehension', 'Argument Comprehension', 'GPT4.1')
GPT4point1_IM_TP += TP
GPT4point1_IM_FN += FN
GPT4point1_IM_FP += FP

#Well-Formedness
TP , FN , FP = calculate_LLM('IM Software', 'Well-Formedness(Syntax)', 'Well-Formedness (Syntax)', 'GPT4.1')
GPT4point1_IM_TP += TP
GPT4point1_IM_FN += FN
GPT4point1_IM_FP += FP



#Expressive Sufficiency
TP , FN , FP = calculate_LLM('IM Software', 'Expressive Sufficiency ', 'Expressive Sufficiency', 'GPT4.1')
GPT4point1_IM_TP += TP
GPT4point1_IM_FN += FN
GPT4point1_IM_FP += FP


#Argument Criticism
TP , FN , FP = calculate_LLM('IM Software', 'Argument Criticism and Defeat', 'Argument Criticism and Defeat', 'GPT4.1')
GPT4point1_IM_TP += TP
GPT4point1_IM_FN += FN
GPT4point1_IM_FP += FP


#Recall
GPT4point1_IM_recall = GPT4point1_IM_TP / (GPT4point1_IM_TP + GPT4point1_IM_FN)
print("IM Software Recall:", GPT4point1_IM_recall)
print("IM Software Recall Percent:", round(GPT4point1_IM_recall * 100, 2))

#Precision
GPT4point1_IM_precision = GPT4point1_IM_TP / (GPT4point1_IM_TP + GPT4point1_IM_FP)
print("IM Software Precision:", GPT4point1_IM_precision)
print("IM Software Precision Percent:", round(GPT4point1_IM_precision * 100, 2))

#F1 Score
GPT4point1_IM_F1 = 2 * (GPT4point1_IM_precision * GPT4point1_IM_recall) / (GPT4point1_IM_precision + GPT4point1_IM_recall)
print("IM Software F1:", GPT4point1_IM_F1)
print("IM Software F1 Percent:", round(GPT4point1_IM_F1 * 100, 2))


###############GPT4.1 Baidu Apollo
GPT4point1_Baidu_TP = 0
GPT4point1_Baidu_FN = 0
GPT4point1_Baidu_FP = 0

#Argument Comprehension
TP , FN , FP = calculate_LLM('Baidu Apollo', 'Argument Comprehension', 'Argument Comprehension', 'GPT4.1')
GPT4point1_Baidu_TP += TP
GPT4point1_Baidu_FN += FN
GPT4point1_Baidu_FP += FP


#Well-Formedness
TP , FN , FP = calculate_LLM('Baidu Apollo', 'Well-Formedness(Syntax)', 'Well-Formedness (Syntax)', 'GPT4.1')
GPT4point1_Baidu_TP += TP
GPT4point1_Baidu_FN += FN
GPT4point1_Baidu_FP += FP


#Expressive Sufficiency
TP , FN , FP = calculate_LLM('Baidu Apollo', 'Expressive Sufficiency ', 'Expressive Sufficiency', 'GPT4.1')
GPT4point1_Baidu_TP += TP
GPT4point1_Baidu_FN += FN
GPT4point1_Baidu_FP += FP

#Argument Critcism and Defeat
TP , FN , FP = calculate_LLM('Baidu Apollo', 'Argument Criticism and Defeat', 'Argument Criticism and Defeat', 'GPT4.1')
GPT4point1_Baidu_TP += TP
GPT4point1_Baidu_FN += FN
GPT4point1_Baidu_FP += FP

#Recall
GPT4point1_Baidu_recall = GPT4point1_Baidu_TP / (GPT4point1_Baidu_TP + GPT4point1_Baidu_FN)
print("Baidu Apollo Recall:", GPT4point1_Baidu_recall)
print("Baidu Apollo Recall Percent:", round(GPT4point1_Baidu_recall * 100, 2))

#Precision
GPT4point1_Baidu_precision = GPT4point1_Baidu_TP / (GPT4point1_Baidu_TP + GPT4point1_Baidu_FP)
print("Baidu Apollo Precision:", GPT4point1_Baidu_precision)
print("Baidu Apollo Precision Percent:", round(GPT4point1_Baidu_precision * 100, 2))

#F1 Score
GPT4point1_Baidu_F1 = 2 * (GPT4point1_Baidu_precision * GPT4point1_Baidu_recall) / (GPT4point1_Baidu_precision + GPT4point1_Baidu_recall)
print("Baidu Apollo F1:", GPT4point1_Baidu_F1)
print("Baidu Apollo F1 Percent:", round(GPT4point1_Baidu_F1 * 100, 2))


###############GPT4.1 Lane Management System
GPT4point1_LMS_TP = 0
GPT4point1_LMS_FN = 0
GPT4point1_LMS_FP = 0



#Argument Comprehension
TP , FN , FP = calculate_LLM('Lane Management System', 'Argument Comprehension', 'Argument Comprehension', 'GPT4.1')
GPT4point1_LMS_TP += TP
GPT4point1_LMS_FN += FN
GPT4point1_LMS_FP += FP



#Well-Formedness
TP , FN , FP = calculate_LLM('Lane Management System', 'Well-Formedness(Syntax)', 'Well-Formedness (Syntax)', 'GPT4.1')
GPT4point1_LMS_TP += TP
GPT4point1_LMS_FN += FN
GPT4point1_LMS_FP += FP


#Expressive Sufficiency
TP , FN , FP = calculate_LLM('Lane Management System', 'Expressive Sufficiency ', 'Expressive Sufficiency', 'GPT4.1')
GPT4point1_LMS_TP += TP
GPT4point1_LMS_FN += FN
GPT4point1_LMS_FP += FP



#Argument Critcism and Defeat
TP , FN , FP = calculate_LLM('Lane Management System', 'Argument Criticism and Defeat', 'Argument Criticism and Defeat', 'GPT4.1')
GPT4point1_LMS_TP += TP
GPT4point1_LMS_FN += FN
GPT4point1_LMS_FP += FP

#Recall
GPT4point1_LMS_recall = GPT4point1_LMS_TP / (GPT4point1_LMS_TP + GPT4point1_LMS_FN)
print("LMS Recall:", GPT4point1_LMS_recall)
print("LMS Recall Percent:", round(GPT4point1_LMS_recall * 100, 2))


#Precision
GPT4point1_LMS_precision = GPT4point1_LMS_TP / (GPT4point1_LMS_TP + GPT4point1_LMS_FP)
print("LMS Precision:",GPT4point1_LMS_precision)
print("LMS Precision Percent:",round(GPT4point1_LMS_precision * 100, 2))


#F1 Score
GPT4point1_LMS_F1 = 2 * (GPT4point1_LMS_precision * GPT4point1_LMS_recall) / (GPT4point1_LMS_precision + GPT4point1_LMS_recall)
print("LMS F1:", GPT4point1_LMS_F1)
print("LMS F1 Percent:", round(GPT4point1_LMS_F1 * 100, 2))

GPCA Recall: 0.6
GPCA Recall Percent: 60.0
GPCA Precision: 0.43902439024390244
GPCA Precision Percent: 43.9
GPCA F1: 0.5070422535211268
GPCA F1 Percent: 50.7
IM Software Recall: 0.3333333333333333
IM Software Recall Percent: 33.33
IM Software Precision: 0.18518518518518517
IM Software Precision Percent: 18.52
IM Software F1: 0.23809523809523808
IM Software F1 Percent: 23.81
Baidu Apollo Recall: 0.9833333333333333
Baidu Apollo Recall Percent: 98.33
Baidu Apollo Precision: 1.0
Baidu Apollo Precision Percent: 100.0
Baidu Apollo F1: 0.9915966386554621
Baidu Apollo F1 Percent: 99.16
LMS Recall: 0.9166666666666666
LMS Recall Percent: 91.67
LMS Precision: 1.0
LMS Precision Percent: 100.0
LMS F1: 0.9565217391304348
LMS F1 Percent: 95.65


DeepSeek-R1

In [ ]:
###############DeepSeek GPCA
Deepseek_GPCA_TP = 0
Deepseek_GPCA_FN = 0
Deepseek_GPCA_FP = 0


#Argument Comprehension
TP , FN , FP = calculate_LLM('GPCA', 'Argument Comprehension', 'Argument Comprehension', 'Deepseek Reasoner')
Deepseek_GPCA_TP += TP
Deepseek_GPCA_FN += FN
Deepseek_GPCA_FP += FP

#Well-Formedness
TP , FN , FP = calculate_LLM('GPCA', 'Well-Formedness(Syntax)', 'Well-Formedness (Syntax)', 'Deepseek Reasoner')
Deepseek_GPCA_TP += TP
Deepseek_GPCA_FN += FN
Deepseek_GPCA_FP += FP


#Expressive Sufficiency
TP , FN , FP = calculate_LLM('GPCA', 'Expressive Sufficiency ', 'Expressive Sufficiency', 'Deepseek Reasoner')
Deepseek_GPCA_TP += TP
Deepseek_GPCA_FN += FN
Deepseek_GPCA_FP += FP

#Argument Comprehension
TP , FN , FP = calculate_LLM('GPCA', 'Argument Criticism and Defeat', 'Argument Criticism and Defeat', 'Deepseek Reasoner')
Deepseek_GPCA_TP += TP
Deepseek_GPCA_FN += FN
Deepseek_GPCA_FP += FP

#Recall
Deepseek_GPCA_recall = Deepseek_GPCA_TP / (Deepseek_GPCA_TP + Deepseek_GPCA_FN)
print("GPCA DeepSeek Recall:", Deepseek_GPCA_recall)
print("GPCA DeepSeek Recall Percent:", round(Deepseek_GPCA_recall * 100, 2))

#Precision
Deepseek_GPCA_precision = Deepseek_GPCA_TP / (Deepseek_GPCA_TP + Deepseek_GPCA_FP)
print("GPCA DeepSeek Precision:", Deepseek_GPCA_precision)
print("GPCA DeepSeek Precision Percent:", round(Deepseek_GPCA_precision * 100, 2))

#F1 Score
Deepseek_GPCA_F1 = 2 * (Deepseek_GPCA_precision * Deepseek_GPCA_recall) / (Deepseek_GPCA_precision + Deepseek_GPCA_recall)
print("GPCA Deepseek F1:", Deepseek_GPCA_F1)
print("GPCA Deepseek F1 Percent:", round(Deepseek_GPCA_F1 * 100, 2))

###############DeepSeek IM Software
Deepseek_IM_TP = 0
Deepseek_IM_FN = 0
Deepseek_IM_FP = 0

#Argument Comprehension
TP , FN , FP = calculate_LLM('IM Software', 'Argument Comprehension', 'Argument Comprehension', 'Deepseek Reasoner')
Deepseek_IM_TP += TP
Deepseek_IM_FN += FN
Deepseek_IM_FP += FP

#Well-Formedness
TP , FN , FP = calculate_LLM('IM Software', 'Well-Formedness(Syntax)', 'Well-Formedness (Syntax)', 'Deepseek Reasoner')
Deepseek_IM_TP += TP
Deepseek_IM_FN += FN
Deepseek_IM_FP += FP

#Expressive Sufficiency
TP , FN , FP = calculate_LLM('IM Software', 'Expressive Sufficiency ', 'Expressive Sufficiency', 'Deepseek Reasoner')
Deepseek_IM_TP += TP
Deepseek_IM_FN += FN
Deepseek_IM_FP += FP

#Argument Criticism
TP , FN , FP = calculate_LLM('IM Software', 'Argument Criticism and Defeat', 'Argument Criticism and Defeat', 'Deepseek Reasoner')
Deepseek_IM_TP += TP
Deepseek_IM_FN += FN
Deepseek_IM_FP += FP

#Recall
Deepseek_IM_recall = Deepseek_IM_TP / (Deepseek_IM_TP + Deepseek_IM_FN)
print("IM Software DeepSeek Recall:", Deepseek_IM_recall)
print("IM Software DeepSeek Recall Percent:", round(Deepseek_IM_recall * 100, 2))

#Precision
Deepseek_IM_precision = Deepseek_IM_TP / (Deepseek_IM_TP + Deepseek_IM_FP)
print("IM Software Deepseek Precision:", Deepseek_IM_precision)
print("IM Software Deepseek Precision Percent:", round(Deepseek_IM_precision * 100, 2))

#F1 Score
Deepseek_IM_F1 = 2 * (Deepseek_IM_precision * Deepseek_IM_recall) / (Deepseek_IM_precision + Deepseek_IM_recall)
print("IM Software Deepseek F1:", Deepseek_IM_F1)
print("IM Software Deepseek F1 Percent:", round(Deepseek_IM_F1 * 100, 2))

###############DeepSeek Baidu Apollo
Deepseek_Baidu_TP = 0
Deepseek_Baidu_FN = 0
Deepseek_Baidu_FP = 0


#Argument Comprehension
TP , FN , FP = calculate_LLM('Baidu Apollo', 'Argument Comprehension', 'Argument Comprehension', 'Deepseek Reasoner')
Deepseek_Baidu_TP += TP
Deepseek_Baidu_FN += FN
Deepseek_Baidu_FP += FP

#Well-Formedness
TP , FN , FP = calculate_LLM('Baidu Apollo', 'Well-Formedness(Syntax)', 'Well-Formedness (Syntax)', 'Deepseek Reasoner')
Deepseek_Baidu_TP += TP
Deepseek_Baidu_FN += FN
Deepseek_Baidu_FP += FP

#Expressive Sufficiency
TP , FN , FP = calculate_LLM('Baidu Apollo', 'Expressive Sufficiency ', 'Expressive Sufficiency', 'Deepseek Reasoner')
Deepseek_Baidu_TP += TP
Deepseek_Baidu_FN += FN
Deepseek_Baidu_FP += FP

#Argument Criticism and Defeat
TP , FN , FP = calculate_LLM('Baidu Apollo', 'Argument Criticism and Defeat', 'Argument Criticism and Defeat', 'Deepseek Reasoner')
Deepseek_Baidu_TP += TP
Deepseek_Baidu_FN += FN
Deepseek_Baidu_FP += FP

#Recall
Deepseek_Baidu_recall = Deepseek_Baidu_TP / (Deepseek_Baidu_TP + Deepseek_Baidu_FN)
print("Baidu Apollo DeepSeek Recall:", Deepseek_Baidu_recall)
print("Baidu Apollo DeepSeek Recall Percent:", round(Deepseek_Baidu_recall * 100, 2))

#Precision
Deepseek_Baidu_precision = Deepseek_Baidu_TP / (Deepseek_Baidu_TP + Deepseek_Baidu_FP)
print("Baidu Apollo Deepseek Precision:", Deepseek_Baidu_precision)
print("Baidu Apollo Deepseek Precision Percent:", round(Deepseek_Baidu_precision * 100, 2))

#F1 Score
Deepseek_Baidu_F1 = 2 * (Deepseek_Baidu_precision * Deepseek_Baidu_recall) / (Deepseek_Baidu_precision + Deepseek_Baidu_recall)
print("Baidu Apollo Deepseek F1:", Deepseek_Baidu_F1)
print("Baidu Apollo Deepseek F1 Percent:", round(Deepseek_Baidu_F1 * 100, 2))

###############DeepSeek LMS
Deepseek_LMS_TP = 0
Deepseek_LMS_FN = 0
Deepseek_LMS_FP = 0


#Argument Comprehension
TP , FN , FP = calculate_LLM('Lane Management System', 'Argument Comprehension', 'Argument Comprehension', 'Deepseek Reasoner')
Deepseek_LMS_TP += TP
Deepseek_LMS_FN += FN
Deepseek_LMS_FP += FP

#Well-Formedness
TP , FN , FP = calculate_LLM('Lane Management System', 'Well-Formedness(Syntax)', 'Well-Formedness (Syntax)', 'Deepseek Reasoner')
Deepseek_LMS_TP += TP
Deepseek_LMS_FN += FN
Deepseek_LMS_FP += FP

#Expressive Sufficiency
TP , FN , FP = calculate_LLM('Lane Management System', 'Expressive Sufficiency ', 'Expressive Sufficiency', 'Deepseek Reasoner')
Deepseek_LMS_TP += TP
Deepseek_LMS_FN += FN
Deepseek_LMS_FP += FP

#Argument Criticism and Defeat
TP , FN , FP = calculate_LLM('Lane Management System', 'Argument Criticism and Defeat', 'Argument Criticism and Defeat', 'Deepseek Reasoner')
Deepseek_LMS_TP += TP
Deepseek_LMS_FN += FN
Deepseek_LMS_FP += FP

#Recall
Deepseek_LMS_recall = Deepseek_LMS_TP / (Deepseek_LMS_TP + Deepseek_LMS_FN)
print("LMS DeepSeek Recall:", Deepseek_LMS_recall)
print("LMS DeepSeek Recall Percent:", round(Deepseek_LMS_recall * 100, 2))


#Precision
Deepseek_LMS_precision = Deepseek_LMS_TP / (Deepseek_LMS_TP + Deepseek_LMS_FP)
print("LMS Deepseek Precision:", Deepseek_LMS_precision)
print("LMS Deepseek Precision Percent:", round(Deepseek_LMS_precision * 100, 2))

#F1 Score
Deepseek_LMS_F1 = 2 * (Deepseek_LMS_precision * Deepseek_LMS_recall) / (Deepseek_LMS_precision + Deepseek_LMS_recall)
print("LMS Deepseek F1:", Deepseek_LMS_F1)
print("LMS Deepseek F1 Percent:", round(Deepseek_LMS_F1 * 100, 2))

GPCA DeepSeek Recall: 0.26666666666666666
GPCA DeepSeek Recall Percent: 26.67
GPCA DeepSeek Precision: 0.6666666666666666
GPCA DeepSeek Precision Percent: 66.67
GPCA Deepseek F1: 0.3809523809523809
GPCA Deepseek F1 Percent: 38.1
IM Software DeepSeek Recall: 0.2
IM Software DeepSeek Recall Percent: 20.0
IM Software Deepseek Precision: 0.15
IM Software Deepseek Precision Percent: 15.0
IM Software Deepseek F1: 0.17142857142857143
IM Software Deepseek F1 Percent: 17.14
Baidu Apollo DeepSeek Recall: 0.9666666666666667
Baidu Apollo DeepSeek Recall Percent: 96.67
Baidu Apollo Deepseek Precision: 1.0
Baidu Apollo Deepseek Precision Percent: 100.0
Baidu Apollo Deepseek F1: 0.983050847457627
Baidu Apollo Deepseek F1 Percent: 98.31
LMS DeepSeek Recall: 0.21666666666666667
LMS DeepSeek Recall Percent: 21.67
LMS Deepseek Precision: 1.0
LMS Deepseek Precision Percent: 100.0
LMS Deepseek F1: 0.3561643835616438
LMS Deepseek F1 Percent: 35.62


Gemini

In [ ]:
###############Gemini GPCA
Gemini_GPCA_TP = 0
Gemini_GPCA_FN = 0
Gemini_GPCA_FP = 0

#Argument Comprehension
TP , FN , FP = calculate_LLM('GPCA', 'Argument Comprehension', 'Argument Comprehension', 'Gemini 2.0 Flash')
Gemini_GPCA_TP += TP
Gemini_GPCA_FN += FN
Gemini_GPCA_FP += FP

#Well-Formedness
TP , FN , FP = calculate_LLM('GPCA', 'Well-Formedness(Syntax)', 'Well-Formedness (Syntax)', 'Gemini 2.0 Flash')
Gemini_GPCA_TP += TP
Gemini_GPCA_FN += FN
Gemini_GPCA_FP += FP

#Expressive Sufficiency
TP , FN , FP = calculate_LLM('GPCA', 'Expressive Sufficiency ', 'Expressive Sufficiency', 'Gemini 2.0 Flash')
Gemini_GPCA_TP += TP
Gemini_GPCA_FN += FN
Gemini_GPCA_FP += FP

#Argument Criticism and Defeat
TP , FN , FP = calculate_LLM('GPCA', 'Argument Criticism and Defeat', 'Argument Criticism and Defeat', 'Gemini 2.0 Flash')
Gemini_GPCA_TP += TP
Gemini_GPCA_FN += FN
Gemini_GPCA_FP += FP

#Recall
Gemini_GPCA_recall = Gemini_GPCA_TP / (Gemini_GPCA_TP + Gemini_GPCA_FN)
print("Gemini GPCA recall:", Gemini_GPCA_recall)
print("Gemini GPCA recall Percent:", round(Gemini_GPCA_recall * 100, 2))

#Precision
Gemini_GPCA_precision = Gemini_GPCA_TP / (Gemini_GPCA_TP + Gemini_GPCA_FP)
print("Gemini GPCA precision:", Gemini_GPCA_precision)
print("Gemini GPCA precision Percent:", round(Gemini_GPCA_precision * 100, 2))

#F1 Score
Gemini_GPCA_F1 = 2 * (Gemini_GPCA_precision * Gemini_GPCA_recall) / (Gemini_GPCA_precision + Gemini_GPCA_recall)
print("Gemini GPCA F1:", Gemini_GPCA_F1)
print("Gemini GPCA F1 Percent:", round(Gemini_GPCA_F1 * 100, 2))

###############Gemini IM Software
Gemini_IM_TP = 0
Gemini_IM_FN = 0
Gemini_IM_FP = 0

#Argument Comprehension
TP , FN , FP = calculate_LLM('IM Software', 'Argument Comprehension', 'Argument Comprehension', 'Gemini 2.0 Flash')
Gemini_IM_TP += TP
Gemini_IM_FN += FN
Gemini_IM_FP += FP

#Well-Formedness
TP , FN , FP = calculate_LLM('IM Software', 'Well-Formedness(Syntax)', 'Well-Formedness (Syntax)', 'Gemini 2.0 Flash')
Gemini_IM_TP += TP
Gemini_IM_FN += FN
Gemini_IM_FP += FP

#Expressive Sufficiency
TP , FN , FP = calculate_LLM('IM Software', 'Expressive Sufficiency ', 'Expressive Sufficiency', 'Gemini 2.0 Flash')
Gemini_IM_TP += TP
Gemini_IM_FN += FN
Gemini_IM_FP += FP

#Argument Criticism and Defeat
TP , FN , FP = calculate_LLM('IM Software', 'Argument Criticism and Defeat', 'Argument Criticism and Defeat', 'Gemini 2.0 Flash')
Gemini_IM_TP += TP
Gemini_IM_FN += FN
Gemini_IM_FP += FP

#Recall
Gemini_IM_recall = Gemini_IM_TP / (Gemini_IM_TP + Gemini_IM_FN)
print("Gemini IM Software recall:", Gemini_IM_recall)
print("Gemini IM Software recall Percent:", round(Gemini_IM_recall * 100, 2))

#Precision
Gemini_IM_precision = Gemini_IM_TP / (Gemini_IM_TP + Gemini_IM_FP)
print("Gemini IM Software precision:", Gemini_IM_precision)
print("Gemini IM Software precision Percent:", round(Gemini_IM_precision * 100, 2))

#F1 Score
Gemini_IM_F1 = 2 * (Gemini_IM_precision * Gemini_IM_recall) / (Gemini_IM_precision + Gemini_IM_recall)
print("Gemini IM Software F1:", Gemini_IM_F1)
print("Gemini IM Software F1 Percent:", round(Gemini_IM_F1 * 100, 2))

###############Gemini Baidu Apollo
Gemini_Baidu_TP = 0
Gemini_Baidu_FN = 0
Gemini_Baidu_FP = 0

#Argument Comprehension
TP , FN , FP = calculate_LLM('Baidu Apollo', 'Argument Comprehension', 'Argument Comprehension', 'Gemini 2.0 Flash')
Gemini_Baidu_TP += TP
Gemini_Baidu_FN += FN
Gemini_Baidu_FP += FP

#Well-Formedness
TP , FN , FP = calculate_LLM('Baidu Apollo', 'Well-Formedness(Syntax)', 'Well-Formedness (Syntax)', 'Gemini 2.0 Flash')
Gemini_Baidu_TP += TP
Gemini_Baidu_FN += FN
Gemini_Baidu_FP += FP

#Expressive Sufficiency
TP , FN , FP = calculate_LLM('Baidu Apollo', 'Expressive Sufficiency ', 'Expressive Sufficiency', 'Gemini 2.0 Flash')
Gemini_Baidu_TP += TP
Gemini_Baidu_FN += FN
Gemini_Baidu_FP += FP

#Argument Criticism and Defeat
TP , FN , FP = calculate_LLM('Baidu Apollo', 'Argument Criticism and Defeat', 'Argument Criticism and Defeat', 'Gemini 2.0 Flash')
Gemini_Baidu_TP += TP
Gemini_Baidu_FN += FN
Gemini_Baidu_FP += FP

#Recall
Gemini_Baidu_recall = Gemini_Baidu_TP / (Gemini_Baidu_TP + Gemini_Baidu_FN)
print("Gemini Baidu Apollo recall:", Gemini_Baidu_recall)
print("Gemini Baidu Apollo recall Percent:", round(Gemini_Baidu_recall * 100, 2))

#Precision
Gemini_Baidu_precision = Gemini_Baidu_TP / (Gemini_Baidu_TP + Gemini_Baidu_FP)
print("Gemini Baidu Apollo precision:", Gemini_Baidu_precision)
print("Gemini Baidu Apollo precision Percent:", round(Gemini_Baidu_precision * 100, 2))

#F1 Score
Gemini_Baidu_F1 = 2 * (Gemini_Baidu_precision * Gemini_Baidu_recall) / (Gemini_Baidu_precision + Gemini_Baidu_recall)
print("Gemini Baidu Apollo F1:", Gemini_Baidu_F1)
print("Gemini Baidu Apollo F1 Percent:", round(Gemini_Baidu_F1 * 100, 2))

###############Gemini LMS
Gemini_LMS_TP = 0
Gemini_LMS_FN = 0
Gemini_LMS_FP = 0

#Argument Comprehension
TP , FN , FP = calculate_LLM('Lane Management System', 'Argument Comprehension', 'Argument Comprehension', 'Gemini 2.0 Flash')
Gemini_LMS_TP += TP
Gemini_LMS_FN += FN
Gemini_LMS_FP += FP

#Well-Formedness
TP , FN , FP = calculate_LLM('Lane Management System', 'Well-Formedness(Syntax)', 'Well-Formedness (Syntax)', 'Gemini 2.0 Flash')
Gemini_LMS_TP += TP
Gemini_LMS_FN += FN
Gemini_LMS_FP += FP

#Expressive Sufficiency
TP , FN , FP = calculate_LLM('Lane Management System', 'Expressive Sufficiency ', 'Expressive Sufficiency', 'Gemini 2.0 Flash')
Gemini_LMS_TP += TP
Gemini_LMS_FN += FN
Gemini_LMS_FP += FP

#Argument Criticism and Defeat
TP , FN , FP = calculate_LLM('Lane Management System', 'Argument Criticism and Defeat', 'Argument Criticism and Defeat', 'Gemini 2.0 Flash')
Gemini_LMS_TP += TP
Gemini_LMS_FN += FN
Gemini_LMS_FP += FP

#Recall
Gemini_LMS_recall = Gemini_LMS_TP / (Gemini_LMS_TP + Gemini_LMS_FN)
print("Gemini LMS recall:", Gemini_LMS_recall)
print("Gemini LMS recall Percent:", round(Gemini_LMS_recall * 100, 2))

#Precision
Gemini_LMS_precision = Gemini_LMS_TP / (Gemini_LMS_TP + Gemini_LMS_FP)
print("Gemini LMS precision:", Gemini_LMS_precision)
print("Gemini LMS precision Percent:", round(Gemini_LMS_precision * 100, 2))

#F1 Score
Gemini_LMS_F1 = 2 * (Gemini_LMS_precision * Gemini_LMS_recall) / (Gemini_LMS_precision + Gemini_LMS_recall)
print("Gemini LMS F1:", Gemini_LMS_F1)
print("Gemini LMS F1 Percent:", round(Gemini_LMS_F1 * 100, 2))

Gemini GPCA recall: 0.7333333333333333
Gemini GPCA recall Percent: 73.33
Gemini GPCA precision: 0.4888888888888889
Gemini GPCA precision Percent: 48.89
Gemini GPCA F1: 0.5866666666666667
Gemini GPCA F1 Percent: 58.67
Gemini IM Software recall: 0.8
Gemini IM Software recall Percent: 80.0
Gemini IM Software precision: 0.27906976744186046
Gemini IM Software precision Percent: 27.91
Gemini IM Software F1: 0.4137931034482758
Gemini IM Software F1 Percent: 41.38
Gemini Baidu Apollo recall: 0.9666666666666667
Gemini Baidu Apollo recall Percent: 96.67
Gemini Baidu Apollo precision: 1.0
Gemini Baidu Apollo precision Percent: 100.0
Gemini Baidu Apollo F1: 0.983050847457627
Gemini Baidu Apollo F1 Percent: 98.31
Gemini LMS recall: 0.8333333333333334
Gemini LMS recall Percent: 83.33
Gemini LMS precision: 1.0
Gemini LMS precision Percent: 100.0
Gemini LMS F1: 0.9090909090909091
Gemini LMS F1 Percent: 90.91
